# FairScope: On-Device Model Quantization & TFLite Conversion Pipeline
**Member C:** Edge AI Deployment, Model Compression & Mobile Optimization
**SLIIT Research Project:** J26-IT-371 | **Date:** October 2026

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chanuka001/Fairscope_research/blob/feat/monal/app-shell/models/compressed/quantization_pipeline.ipynb)

---

### 📖 Purpose
This Google Colab notebook converts Member A's trained dermatological skin classification model (`model.onnx`) into the two edge-optimized TensorFlow Lite (`.tflite`) formats required by the FairScope Flutter mobile app:
1. **`baseline_fp32.tflite`**: Full 32-bit floating-point uncompressed baseline (~8–14 MB).
2. **`quantized_int8.tflite`**: 8-bit integer post-training quantized model (~2–4 MB, ~70%+ size reduction, low-latency mobile CPU execution).
3. **`labels.txt`**: 5-class target dermatology labels:
   - `Acne`
   - `Benign Nevus`
   - `Eczema`
   - `Melanoma`
   - `Psoriasis`

### Step 1: Install Conversion Tools & NumPy Fix
Installs `numpy<2.0.0` (required for TensorFlow compatibility), `onnx`, `onnx2tf`, and visualization tools.

> **Note:** If you just installed `numpy<2.0.0` for the first time, click **Restart** (or **Runtime → Restart session**) in your notebook toolbar before running the next cells.

In [1]:
!pip install -q "numpy<2.0.0" onnx onnx2tf tensorflow matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 87.9 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 kB 20.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 kB 20.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 kB 20.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 kB 19.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 kB 19.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 kB 19.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 kB 19.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 kB 19.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 kB 19.8 MB/s eta 0:00:00
  

### Step 2: Locate or Upload `model.onnx`
Automatically detects `model.onnx` from local repository paths (or prompts for upload if in remote Colab).

In [ ]:
import os
import shutil

# Check candidate paths where model.onnx might live
candidate_paths = [
    "model.onnx",
    "../image_branch/model.onnx",
    "models/image_branch/model.onnx",
    "../../models/image_branch/model.onnx",
]

found = False
for p in candidate_paths:
    if os.path.exists(p):
        if p != "model.onnx":
            shutil.copyfile(p, "model.onnx")
        print(f"✅ Found model.onnx at: {os.path.abspath(p)}")
        found = True
        break

if not found:
    try:
        from google.colab import files
        print("📂 Please upload your 'model.onnx' file:")
        files.upload()
    except Exception:
        pass

assert os.path.exists("model.onnx"), "Error: model.onnx not found! Please place model.onnx in this folder."
print(f"✅ Ready: model.onnx ({os.path.getsize('model.onnx') / (1024*1024):.2f} MB)")

### Step 3: Inspect ONNX Model Graph & Architecture
Inspects input dimensions, output classes, and operations.

In [ ]:
import onnx

model = onnx.load("model.onnx")
print("=== Model Architecture & Metadata ===")
print(f"Producer: {model.producer_name} (v{model.producer_version})")

for inp in model.graph.input:
    shape = [d.dim_value if d.dim_value != 0 else d.dim_param for d in inp.type.tensor_type.shape.dim]
    print(f"Input: {inp.name} | Shape: {shape} | Type: {inp.type.tensor_type.elem_type}")

for out in model.graph.output:
    shape = [d.dim_value if d.dim_value != 0 else d.dim_param for d in out.type.tensor_type.shape.dim]
    print(f"Output: {out.name} | Shape: {shape} | Type: {out.type.tensor_type.elem_type}")

num_nodes = len(model.graph.node)
print(f"Total Graph Operators: {num_nodes}")

### Step 4: Convert ONNX to TensorFlow & TFLite (FP32 & INT8)
Converts the PyTorch/ONNX graph into:
- `baseline_fp32.tflite`
- `quantized_int8.tflite`

In [ ]:
import os
import shutil

output_dir = "tflite_models"
os.makedirs(output_dir, exist_ok=True)

fp32_output_path = os.path.join(output_dir, "baseline_fp32.tflite")
int8_output_path = os.path.join(output_dir, "quantized_int8.tflite")

print("🔄 Running onnx2tf conversion...")
# onnx2tf runs in an isolated subshell with full ONNX-to-TFLite translation
!onnx2tf -in model.onnx -coopt -cmod -output_integer_quantized_tflite

# Locate generated tflite files from saved_model folder
saved_model_dir = "saved_model"
if os.path.exists(saved_model_dir):
    for fname in os.listdir(saved_model_dir):
        full_path = os.path.join(saved_model_dir, fname)
        if fname.endswith("float32.tflite") or (fname.endswith(".tflite") and "int8" not in fname and "quant" not in fname):
            shutil.copyfile(full_path, fp32_output_path)
            print(f"✅ Copied FP32 model to {fp32_output_path}")
        elif "integer_quant" in fname or "int8" in fname:
            shutil.copyfile(full_path, int8_output_path)
            print(f"✅ Copied INT8 model to {int8_output_path}")

# Fallback: If onnx2tf produced saved_model but not INT8, run converter
if not os.path.exists(int8_output_path) and os.path.exists(saved_model_dir):
    try:
        import numpy as np
        import tensorflow as tf
        print("🔄 Applying Post-Training INT8 Quantization...")
        converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        def representative_dataset():
            for _ in range(50):
                data = np.random.uniform(-1.0, 1.0, size=(1, 224, 224, 3)).astype(np.float32)
                yield [data]
        converter.representative_dataset = representative_dataset
        int8_model = converter.convert()
        with open(int8_output_path, "wb") as f:
            f.write(int8_model)
        print(f"✅ Generated {int8_output_path}")
    except Exception as e:
        print(f"Note: Converter fallback skipped: {e}")

# Ensure files exist or report clearly
if os.path.exists(fp32_output_path) and os.path.exists(int8_output_path):
    fp32_size = os.path.getsize(fp32_output_path) / (1024 * 1024)
    int8_size = os.path.getsize(int8_output_path) / (1024 * 1024)
    reduction = (1.0 - (int8_size / fp32_size)) * 100
    print("\n=== CONVERSION SUCCESSFUL ===")
    print(f"Baseline FP32:  {fp32_size:.2f} MB")
    print(f"Quantized INT8: {int8_size:.2f} MB")
    print(f"Storage Gain:   {reduction:.1f}% size reduction ({fp32_size / int8_size:.2f}x smaller)")
else:
    print("⚠️ Check saved_model directory output for generated files.")

### Step 5: Export Clinical Target `labels.txt`
Generates the 5 clinical diagnostic labels matching FairScope's research classes.

In [ ]:
labels = [
    "Acne",
    "Benign Nevus",
    "Eczema",
    "Melanoma",
    "Psoriasis"
]

labels_path = os.path.join(output_dir, "labels.txt")
with open(labels_path, "w", encoding="utf-8") as f:
    for l in labels:
        f.write(f"{l}\n")

print(f"✅ Saved labels to {labels_path}")

### Step 6: Verify Model Inferences on Both TFLite Variants
Runs test inferences through both interpreters to verify valid predictions and tensor types.

In [ ]:
def test_tflite_model(model_path, name):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    
    in_details = interpreter.get_input_details()[0]
    out_details = interpreter.get_output_details()[0]
    
    print(f"\n--- {name} Details ---")
    print(f"Input:  Shape={in_details['shape']} | Dtype={in_details['dtype']}")
    print(f"Output: Shape={out_details['shape']} | Dtype={out_details['dtype']}")
    
    # Generate dummy normalized skin image input
    input_shape = in_details['shape']
    if in_details['dtype'] == np.int8:
        dummy_input = np.random.randint(-128, 127, size=input_shape, dtype=np.int8)
    else:
        dummy_input = np.random.uniform(-1.0, 1.0, size=input_shape).astype(np.float32)
        
    interpreter.set_tensor(in_details['index'], dummy_input)
    interpreter.invoke()
    output = interpreter.get_tensor(out_details['index'])
    print(f"Inference successful! Raw Output Logits: {output}")

test_tflite_model(fp32_output_path, "Baseline FP32")
test_tflite_model(int8_output_path, "Quantized INT8")

### Step 7: Plot Compression Visualizer Chart
Generates a chart for your research presentation and progress reports.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
categories = ['Baseline FP32', 'Quantized INT8']
sizes = [fp32_size, int8_size]
colors = ['#0284C7', '#059669']

bars = ax.bar(categories, sizes, color=colors, width=0.45)
ax.set_ylabel('Model Footprint (MB)', fontsize=11, fontweight='bold')
ax.set_title('FairScope Edge Model: Float32 vs INT8 Footprint', fontsize=12, fontweight='bold', pad=12)
ax.set_ylim(0, max(sizes) * 1.3)

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, h + 0.3, f"{h:.2f} MB", ha='center', va='bottom', fontweight='bold')

chart_path = "quantization_comparison.png"
plt.tight_layout()
plt.savefig(chart_path)
plt.show()
print(f"✅ Saved chart to {chart_path}")

### Step 8: Download `.tflite` Models & Labels
Downloads the generated files directly to your computer so you can copy them into `app/assets/models/`.

In [ ]:
from google.colab import files

print("📥 Downloading converted files for FairScope app...")
files.download(fp32_output_path)
files.download(int8_output_path)
files.download(labels_path)
files.download(chart_path)
print("✅ All downloads initiated! Place baseline_fp32.tflite and quantized_int8.tflite into app/assets/models/")